## 2. Own analysis

In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.base import clone
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score

# ==========================================
# 1. DATA PREPARATION & SETUP
# ==========================================

data = pd.read_csv('data/SP 500 ESG Risk Ratings.csv')

# Convert the entire column to string to handle mixed types
data['Full Time Employees'] = data['Full Time Employees'].astype(str)

# Remove commas
data['Full Time Employees'] = data['Full Time Employees'].str.replace(',', '')

# Convert back to float
data['Full Time Employees'] = data['Full Time Employees'].astype(float)

# Fill NA values with the median
data['Full Time Employees'].fillna(data['Full Time Employees'].median(), inplace=True)

data.dropna(subset=['Total ESG Risk score', 'Environment Risk Score', 'Governance Risk Score', 'Social Risk Score'], inplace=True)

# Fill 'Full Time Employees' with the median value
data['Full Time Employees'].fillna(data['Full Time Employees'].median(), inplace=True)

# Fill categorical variables like 'Controversy Level' and 'ESG Risk Level' with 'Unknown'
data['Controversy Level'].fillna('Unknown', inplace=True)
data['ESG Risk Level'].fillna('Unknown', inplace=True)

# Check if 'ESG Risk Percentile' is object type and clean it
if data['ESG Risk Percentile'].dtype == 'object':
    data['ESG Risk Percentile'] = data['ESG Risk Percentile'].str.extract('(\d+)').astype(float)



# Select Features
X = data[['Sector', 'Industry', 'Full Time Employees', 'Environment Risk Score', 
          'Governance Risk Score', 'Social Risk Score', 'Controversy Level', 'Controversy Score']]
y = data['ESG Risk Level']

# Define Categorical Columns (Crucial for XGBoost 'hist' mode)
cat_cols = ['Sector', 'Industry', 'Controversy Level']
for col in cat_cols:
    X[col] = X[col].astype('category')

# Encode Target
le = LabelEncoder()
y_encoded = le.fit_transform(y)

# Split Data (Hold out 20% for final unseen testing)
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, stratify=y_encoded, random_state=42
)

print(f"Training Shape: {X_train.shape}")
print(f"Testing Shape:  {X_test.shape}")

# ==========================================
# 2. MODEL CONFIGURATION 
# ==========================================

# Place your optimized parameters here
hyperopt_params = {
    'n_estimators': 200,
    'learning_rate': 0.1,
    'max_depth': 6,
    'min_child_weight': 1,
    'gamma': 0.1,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
}

# Base Model Definition
base_model = xgb.XGBClassifier(
    **hyperopt_params,
    objective='multi:softprob', 
    tree_method='hist',          # Faster and supports categories
    enable_categorical=True,     # Native categorical support
    eval_metric='mlogloss',
    random_state=42
)

# ==========================================
# ENSEMBLE 
# ==========================================

models = [] # This will store our 5 trained models
cv_scores = []
kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print("\n--- Starting Ensemble Training (5 Folds) ---")

for fold_idx, (train_idx, val_idx) in enumerate(kfold.split(X_train, y_train)):
    
    # 1. Split X_train into Fold Train and Fold Validation
    X_fold_train, y_fold_train = X_train.iloc[train_idx], y_train[train_idx]
    X_fold_val, y_fold_val = X_train.iloc[val_idx], y_train[val_idx]
    
    # 2. Clone the base model (creates a fresh, untrained copy)
    model_instance = clone(base_model)
    
    # 3. Train this instance
    model_instance.fit(
        X_fold_train, y_fold_train,
        eval_set=[(X_fold_val, y_fold_val)],
        verbose=False # Set to True if you want to see training logs per fold
    )
    
    # 4. Check accuracy for this fold (Internal Check)
    val_pred = model_instance.predict(X_fold_val)
    score = accuracy_score(y_fold_val, val_pred)
    cv_scores.append(score)
    
    # 5. Save the trained model to our list
    models.append(model_instance)
    print(f"Fold {fold_idx+1} Accuracy: {score:.4f}")

print(f"Mean Ensemble Validation Accuracy: {np.mean(cv_scores):.4f}")

# ==========================================
#PREDICTION 
# ==========================================

def predict_ensemble(new_data, models, label_encoder, categorical_cols=None):
    """
    Predicts using the average probability of all 5 models (Soft Voting).
    Returns a DataFrame with the readable Label and Confidence score.
    """
    # 1. Prepare Data (Avoid modifying original)
    X_temp = new_data.copy()
    
    # Ensure categorical columns match the training format
    if categorical_cols:
        for col in categorical_cols:
            if col in X_temp.columns:
                X_temp[col] = X_temp[col].astype('category')
    
    # 2. Initialize Probability Matrix
    n_classes = len(label_encoder.classes_)
    avg_probs = np.zeros((X_temp.shape[0], n_classes))
    
    # 3. Aggregate Probabilities from all models
    for model in models:
        avg_probs += model.predict_proba(X_temp)
        
    # 4. Calculate Average
    avg_probs /= len(models)
    
    # 5. Determine Winning Class
    y_indices = np.argmax(avg_probs, axis=1) # Index of highest prob
    max_probs = np.max(avg_probs, axis=1)    # Value of highest prob
    
    # Decode to String Label
    y_labels = label_encoder.inverse_transform(y_indices)
    
    # 6. Format Output
    results = pd.DataFrame({
        'Predicted_Label': y_labels,
        'Confidence': max_probs
    })
    
    # Attach raw probabilities for advanced analysis
    for i, class_name in enumerate(label_encoder.classes_):
        results[f'Prob_{class_name}'] = avg_probs[:, i]
        
    return results

# ==========================================
# EVALUATION
# ==========================================

print("\n--- Evaluating Ensemble on Test Set ---")

# Generate predictions using our function
results_df = predict_ensemble(X_test, models, le, cat_cols)

# Extract predicted labels and probabilities for metrics
y_pred_labels = results_df['Predicted_Label'].values
# Re-encode predictions to numbers for accuracy_score
y_pred_encoded = le.transform(y_pred_labels) 
# Get probability columns for ROC AUC
prob_cols = [col for col in results_df.columns if col.startswith('Prob_')]
y_pred_proba = results_df[prob_cols].values

# Metrics
accuracy = accuracy_score(y_test, y_pred_encoded)
print(f"Final Test Accuracy: {accuracy:.4f}")

print("\nClassification Report:")
print(classification_report(le.inverse_transform(y_test), y_pred_labels))

try:
    roc_auc = roc_auc_score(y_test, y_pred_proba, multi_class='ovr', average='weighted')
    print(f"ROC AUC Score (Weighted OvR): {roc_auc:.4f}")
except Exception as e:
    print(f"Could not calculate ROC AUC: {e}")

print("\nSample Predictions:")
print(results_df[['Predicted_Label', 'Confidence']].head())

Training Shape: (344, 8)
Testing Shape:  (86, 8)

--- Starting Ensemble Training (5 Folds) ---
Fold 1 Accuracy: 0.8406
Fold 2 Accuracy: 0.8551
Fold 3 Accuracy: 0.8261
Fold 4 Accuracy: 0.8986
Fold 5 Accuracy: 0.8529
Mean Ensemble Validation Accuracy: 0.8546

--- Evaluating Ensemble on Test Set ---
Final Test Accuracy: 0.8605

Classification Report:
              precision    recall  f1-score   support

        High       0.80      0.80      0.80        10
         Low       0.89      0.89      0.89        37
      Medium       0.85      0.89      0.87        37
  Negligible       0.00      0.00      0.00         1
      Severe       0.00      0.00      0.00         1

    accuracy                           0.86        86
   macro avg       0.51      0.52      0.51        86
weighted avg       0.84      0.86      0.85        86

ROC AUC Score (Weighted OvR): 0.9689

Sample Predictions:
  Predicted_Label  Confidence
0             Low    0.970082
1          Medium    0.733648
2            